# faiss GPU

* create an enviorment with python 3.10
* conda install nvidia/label/cuda-12.2.2::cuda-toolkit
* conda install faiss-gpu -c pytorch

In [2]:
# import
import faiss

ImportError: /home/sho/anaconda3/envs/faiss-gpu/lib/python3.10/site-packages/faiss/../../../libfaiss.so: undefined symbol: __libc_single_threaded

In [ ]:

def faiss_search_gpu(df, top_k=10, index_type='flat', nlist=100, nprobe=10):
    embedding_matrix = np.stack(df['embedding'].values).astype(np.float32)
    dim = embedding_matrix.shape[1]
    gpu_res = faiss.StandardGpuResources()
    index = None

    # --- Build CPU base index and transfer to GPU ---
    if index_type == 'flat':
        base_index = faiss.IndexFlatL2(dim)
        index = faiss.IndexIDMap(base_index)
        index = faiss.index_cpu_to_gpu(gpu_res, 0, index)

    elif index_type == 'ivf':
        quantizer = faiss.IndexFlatL2(dim)
        base_index = faiss.IndexIVFFlat(quantizer, dim, nlist)
        if not base_index.is_trained:
            base_index.train(embedding_matrix)
        base_index.nprobe = nprobe
        index = faiss.IndexIDMap(base_index)
        index = faiss.index_cpu_to_gpu(gpu_res, 0, index)

    else:
        raise ValueError("index_type must be 'flat' or 'ivf'")

    # add embeddings with IDs
    index.add_with_ids(embedding_matrix, np.arange(len(embedding_matrix)))

    top_1_pos, top_3_pos, top_10_pos = 0, 0, 0
    top_1_failures, top_3_failures, top_10_failures = [], [], []
    search_time = []
    total = 0

    for query_idx in tqdm(range(len(df)), desc=f"FAISS-GPU-{index_type.upper()} Exclude-Self"):
        if query_idx >= 100:
            break

        query_vector = embedding_matrix[query_idx].reshape(1, -1)
        query_person_id = df.iloc[query_idx]['ID']
        query_name = df.iloc[query_idx]['name']

        if df[df.ID == query_person_id].shape[0] == 1:
            continue

        total += 1

        # remove query from df
        index.remove_ids(np.array([query_idx], dtype=np.int64))

        start_time = datetime.now()
        _, retrieved_ids = index.search(query_vector, top_k + 5)

        filtered_ids = [i for i in retrieved_ids[0] if i != query_idx][:top_k]
        top_k_ids = df.iloc[filtered_ids]['ID'].tolist()

        # top k
        if top_k_ids and top_k_ids[0] == query_person_id:
            top_1_pos += 1
        else:
            top_1_failures.append({
                'query_idx': query_idx,
                'query_id': query_person_id,
                'query_name': query_name,
                'top_k_ids': top_k_ids[:1]
            })

        end_time = datetime.now()
        search_time.append((end_time - start_time).total_seconds())

        if query_person_id in top_k_ids[:3]:
            top_3_pos += 1
        else:
            top_3_failures.append({
                'query_idx': query_idx,
                'query_id': query_person_id,
                'query_name': query_name,
                'top_k_ids': top_k_ids[:3]
            })

        if query_person_id in top_k_ids[:10]:
            top_10_pos += 1
        else:
            top_10_failures.append({
                'query_idx': query_idx,
                'query_id': query_person_id,
                'query_name': query_name,
                'top_k_ids': top_k_ids
            })

        # Re-add query back to the GPU index
        index.add_with_ids(query_vector, np.array([query_idx], dtype=np.int64))

    # Report
    top1_acc = top_1_pos / total
    top3_acc = top_3_pos / total
    top10_acc = top_10_pos / total
    median_latency = np.median(search_time)

    print(f"Total queries evaluated: {total}")
    print(f"Top-1 Accuracy: {top1_acc:.4f}")
    print(f"Top-3 Accuracy: {top3_acc:.4f}")
    print(f"Top-10 Accuracy: {top10_acc:.4f}")
    print(f"Top-1 Failures: {len(top_1_failures)}")
    print(f"Top-3 Failures: {len(top_3_failures)}")
    print(f"Top-10 Failures: {len(top_10_failures)}")
    print(f"Median Query Time: {median_latency:.4f} seconds")

    return {
        'top1_acc': top1_acc,
        'top3_acc': top3_acc,
        'top10_acc': top10_acc,
        'median_latency': median_latency,
        'top1_failures': top_1_failures,
        'top3_failures': top_3_failures,
        'top10_failures': top_10_failures
    }

In [ ]:
results = faiss_search_gpu(df, top_k=10, index_type='flat')


In [ ]:
results = faiss_search_gpu(df, top_k=10, index_type='ivf', nlist=256, nprobe=16)